# Phase 1: Early Development

This notebook contains the original scripts used for processing the sample dataset and scaling up to the full dataset. It includes the initial secondary calibration techniques that were later revised after mentor feedback.

## Sample Dataset Processing

The initial script used to prove the concept on a small subset of classifications.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import json

# Importing CSV datasets only with required columns
cols_to_use = ['classification_id', 'user_name', 'user_ip', 'annotations', 'subject_data']
df1 = pd.read_csv('sunspot-detectives-classifications_temp.csv', usecols=cols_to_use)
df2 = pd.read_csv('sunspot-detectives-classifications_temp_2.csv', usecols=cols_to_use)

# Merging datasets to remove duplicates and renumbering
df = pd.concat([df1, df2]).drop_duplicates(subset='classification_id')
df = df.reset_index(drop=True)

# Removing rows where annotations and subject_data columns are empty
df = df.dropna(subset=['annotations', 'subject_data'])

# Reassigning database to the 4 columns needed
df = df[['user_name', 'user_ip', 'annotations', 'subject_data']]

# Extracting spot count from annotations column
def extract_count(annotations):
    try:
        result = json.loads(annotations)
        val = result[0]['value'][0]['value']
        if isinstance(val, int):
            return val
        label = result[0]['value'][0]['label']
        return int(label)
    
    except (ValueError, KeyError, TypeError, IndexError, json.JSONDecodeError):
        return None

# Storing spot count data in new column
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)

# Extracting subject data column
def extract_filename(subject_data):
    try:
        result = json.loads(subject_data)
        first_val = list(result.values())[0]
        filename = first_val['Filename']
        name = filename.replace('.png', '')
        parts = name.split('_')
        return (parts[0], parts[1])
    
    except (ValueError, KeyError, TypeError, json.JSONDecodeError, IndexError):
        return (None, None)

# Storing Day ID and Group ID data in 2 new columns
df[['day_id', 'group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)

# Extracting volunteer identity
def assign_volunteer_id(row):
    user_name = row['user_name']
    user_ip = row['user_ip']
    # Case 1 & 2: user_name exists
    if pd.notna(user_name):
        return user_name
    # Case 3: User_name is NaN, fallback to IP
    if pd.notna(user_ip):
        return 'anon_' + str(user_ip)
    # Case 4: Both missing
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

# Dropping now-unnecessary columns
df = df.drop(columns=['annotations', 'subject_data', 'user_name', 'user_ip'])

# Separating teolixx from all others
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

# Resolving duplicates by averaging (using median to ensure it remains whole number)
teolixx = teolixx.groupby(['day_id', 'group_id'])['spot_count'].median().reset_index()
teolixx.columns = ['day_id', 'group_id', 'spot_count']

# Merging other volunteers' counts with teolixx's counts on same images
merged = pd.merge(
    others,
    teolixx[['day_id', 'group_id', 'spot_count']],
    on=['day_id', 'group_id'],
    suffixes=('_volunteer', '_teolixx')
)

# Difference between volunteer's count and teolixx's count
merged['diff'] = merged['spot_count_volunteer'] - merged['spot_count_teolixx']

# Computing per-volunteer bias, scatter, and number of overlapping images
stats = merged.groupby('volunteer_id')['diff'].agg(['mean', 'std', 'count']).reset_index()
stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap']

# Filling NaN scatter values
stats['scatter'] = stats['scatter'].fillna(0)

# Setting thresholds: can be adjusted
MIN_OVERLAP = 5    # Must have counted at least this many images teolixx also counted
MAX_SCATTER = 10   # Standard deviation of difference vs teolixx must be below this

# Considering volunteers in range of thresholds
accepted_volunteers = stats[
    (stats['n_overlap'] >= MIN_OVERLAP) &
    (stats['scatter'] <= MAX_SCATTER)
]['volunteer_id'].tolist()

# Always including teolixx
accepted_volunteers.append('teolixx')

# Volunteers with no overlap with teolixx
no_overlap = set(others['volunteer_id'].unique()) - set(stats['volunteer_id'].unique())

# Filtering data to accepted volunteers only
df_clean = df[df['volunteer_id'].isin(accepted_volunteers)].copy()

# Collapsing within-volunteer duplicates by averaging
df_clean = (
    df_clean
    .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
    .mean()
    .reset_index()
)

# Aggregating spot counts across volunteers per group
group_counts = (
    df_clean
    .groupby(['day_id', 'group_id'])['spot_count']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
group_counts.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']

# Filling undefined std for single-volunteer groups
group_counts['std_count'] = group_counts['std_count'].fillna(0)

# Summing groups to get daily counts with propagated uncertainty
daily = (
    group_counts
    .groupby('day_id')
    .agg(
        daily_count=('mean_count', 'sum'),
        daily_uncertainty=('std_count', lambda x: np.sqrt((x**2).sum())),
        n_groups=('mean_count', 'count')
    )
    .reset_index()
)

# Flagging high-uncertainty days
daily['uncertainty_fraction'] = daily['daily_uncertainty'] / daily['daily_count'].replace(0, np.nan)
daily['quality_flag'] = daily['uncertainty_fraction'].apply(
    lambda x: 'high_uncertainty' if x > 0.5 else ('low_coverage' if pd.isna(x) else 'ok')
)

# Saving outputs
daily.to_csv('daily_sunspot_numbers.csv', index=False)
group_counts.to_csv('group_sunspot_numbers.csv', index=False)
stats.to_csv('volunteer_stats.csv', index=False)

# ── PIPELINE SUMMARY ──────────────────────────────────────────────────────────

# Volunteer quality control
print("VOLUNTEER QUALITY CONTROL")
print(stats.sort_values('scatter', ascending=False).to_string(index=False))
print()
print(f"Accepted:                    {len(accepted_volunteers) - 1}")
print(f"Rejected (bad stats):        {len(stats) - (len(accepted_volunteers) - 1)}")
print(f"No teolixx overlap:          {len(no_overlap)}")
print(f"Total non-teolixx volunteers:{len(others['volunteer_id'].unique())}")

# Coverage
print()
print("COVERAGE")
print(f"Total days:                  {daily['day_id'].nunique()}")
print(f"Total (day, group) pairs:    {len(group_counts)}")
print(f"Single-volunteer groups:     {(group_counts['n_volunteers'] == 1).sum()} / {len(group_counts)} ({100 * (group_counts['n_volunteers'] == 1).mean():.1f}%)")
print(f"Mean volunteers per group:   {group_counts['n_volunteers'].mean():.2f}")

# Daily count statistics
print()
print("DAILY COUNT STATISTICS")
print(daily['daily_count'].describe().round(2).to_string())
print(f"Zero-count days:             {(daily['daily_count'] == 0).sum()}")
print(f"High-uncertainty days:       {(daily['quality_flag'] == 'high_uncertainty').sum()} ({100 * (daily['quality_flag'] == 'high_uncertainty').mean():.1f}%)")

# Worst high-uncertainty days
print()
print("HIGH-UNCERTAINTY DAYS")
print(daily[daily['quality_flag'] == 'high_uncertainty']
      .sort_values('uncertainty_fraction', ascending=False)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'uncertainty_fraction']]
      .to_string(index=False))

# Spot check: highest count days
print()
print("TOP 10 HIGHEST COUNT DAYS")
print(daily.sort_values('daily_count', ascending=False)
      .head(10)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'quality_flag']]
      .to_string(index=False))

# Outputs saved
print()
print("OUTPUTS")
print("Saved: daily_sunspot_numbers.csv")
print("Saved: group_sunspot_numbers.csv")
print("Saved: volunteer_stats.csv")


## Full Dataset Processing (Iterative Builds)

These scripts show the progression of scaling to the 1GB dataset, handling missing records, and attempting secondary calibration for unverified volunteers.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os

# Importing CSV dataset
cols_to_use = ['classification_id', 'user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv('sunspot-detectives-classifications.csv', usecols=cols_to_use)

# Removing rows where annotations and subject_data columns are empty
df = df.dropna(subset=['annotations', 'subject_data'])

# Reassigning database to the 4 columns needed
df = df[['user_name', 'user_ip', 'annotations', 'subject_data']]

# Extracting spot count from annotations column
def extract_count(annotations):
    try:
        result = json.loads(annotations)
        val = result[0]['value'][0]['value']
        if isinstance(val, int):
            return val
        label = result[0]['value'][0]['label']
        return int(label)
    except (ValueError, KeyError, TypeError, IndexError, json.JSONDecodeError):
        return None

# Storing spot count data in new column
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)

# Extracting subject data column
def extract_filename(subject_data):
    try:
        result = json.loads(subject_data)
        first_val = list(result.values())[0]
        filename = first_val['Filename']
        name = filename.replace('.png', '')
        parts = name.split('_')
        return (parts[0], parts[1])
    except (ValueError, KeyError, TypeError, json.JSONDecodeError, IndexError):
        return (None, None)

# Storing Day ID and Group ID data in 2 new columns
df[['day_id', 'group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)

# Extracting volunteer identity
def assign_volunteer_id(row):
    user_name = row['user_name']
    user_ip = row['user_ip']
    if pd.notna(user_name):
        return user_name
    if pd.notna(user_ip):
        return 'anon_' + str(user_ip)
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

# Dropping now-unnecessary columns
df = df.drop(columns=['annotations', 'subject_data', 'user_name', 'user_ip'])

# Separating teolixx from all others
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

# Resolving teolixx duplicates by taking median
teolixx = teolixx.groupby(['day_id', 'group_id'])['spot_count'].median().reset_index()
teolixx.columns = ['day_id', 'group_id', 'spot_count']

# Merging other volunteers' counts with teolixx's counts on same images
merged = pd.merge(
    others,
    teolixx[['day_id', 'group_id', 'spot_count']],
    on=['day_id', 'group_id'],
    suffixes=('_volunteer', '_teolixx')
)

# Difference between volunteer's count and teolixx's count
merged['diff'] = merged['spot_count_volunteer'] - merged['spot_count_teolixx']

# Computing per-volunteer bias, scatter, and number of overlapping images
stats = merged.groupby('volunteer_id')['diff'].agg(['mean', 'std', 'count']).reset_index()
stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap']

# Filling NaN scatter values
stats['scatter'] = stats['scatter'].fillna(0)

# Computing mean teolixx count on each volunteer's overlapping images
teolixx_mean_on_overlap = (
    merged.groupby('volunteer_id')['spot_count_teolixx']
    .mean()
    .reset_index()
)
teolixx_mean_on_overlap.columns = ['volunteer_id', 'teolixx_mean_count']

# Merging into stats
stats = stats.merge(teolixx_mean_on_overlap, on='volunteer_id')

# Computing relative scatter: scatter as fraction of teolixx's mean count on shared images
stats['relative_scatter'] = stats['scatter'] / stats['teolixx_mean_count']

# Setting thresholds: can be adjusted
MIN_OVERLAP = 5             # Must have counted at least this many images teolixx also counted
MAX_RELATIVE_SCATTER = 0.5  # Scatter must be below 50% of teolixx's mean count on shared images

# Considering volunteers in range of thresholds
accepted_volunteers = stats[
    (stats['n_overlap'] >= MIN_OVERLAP) &
    (stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
]['volunteer_id'].tolist()

# Always including teolixx
accepted_volunteers.append('teolixx')

# Volunteers with no overlap with teolixx
no_overlap = set(others['volunteer_id'].unique()) - set(stats['volunteer_id'].unique())

# Filtering data to accepted volunteers only
df_clean = df[df['volunteer_id'].isin(accepted_volunteers)].copy()

# Collapsing within-volunteer duplicates by averaging
df_clean = (
    df_clean
    .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
    .mean()
    .reset_index()
)

# SECONDARY CALIBRATION: no-overlap volunteers vs accepted ensemble

# Build per-group reference mean from accepted volunteers (excluding teolixx)
accepted_no_teolixx = df_clean[df_clean['volunteer_id'] != 'teolixx']
accepted_group_means = (
    accepted_no_teolixx
    .groupby(['day_id', 'group_id'])['spot_count']
    .median()
    .reset_index()
    .rename(columns={'spot_count': 'accepted_mean'})
)

# Raw counts for no-overlap volunteers
no_overlap_df = df[df['volunteer_id'].isin(no_overlap)].copy()

# Merge on shared (day_id, group_id) pairs
merged_no_overlap = pd.merge(
    no_overlap_df,
    accepted_group_means,
    on=['day_id', 'group_id']
)

secondary_accepted = []

if len(merged_no_overlap) > 0:
    merged_no_overlap['diff'] = merged_no_overlap['spot_count'] - merged_no_overlap['accepted_mean']

    no_overlap_stats = (
        merged_no_overlap
        .groupby('volunteer_id')['diff']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    no_overlap_stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap_accepted']
    no_overlap_stats['scatter'] = no_overlap_stats['scatter'].fillna(0)

    accepted_mean_on_overlap = (
        merged_no_overlap
        .groupby('volunteer_id')['accepted_mean']
        .mean()
        .reset_index()
        .rename(columns={'accepted_mean': 'accepted_mean_count'})
    )
    no_overlap_stats = no_overlap_stats.merge(accepted_mean_on_overlap, on='volunteer_id')
    no_overlap_stats['relative_scatter'] = (
        no_overlap_stats['scatter'] / no_overlap_stats['accepted_mean_count']
    )

    secondary_accepted = no_overlap_stats[
        (no_overlap_stats['n_overlap_accepted'] >= MIN_OVERLAP) &
        (no_overlap_stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
    ]['volunteer_id'].tolist()

    # Add secondary-accepted volunteers into df_clean
    df_secondary = df[df['volunteer_id'].isin(secondary_accepted)].copy()
    df_secondary = (
        df_secondary
        .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
        .mean()
        .reset_index()
    )
    df_clean = pd.concat([df_clean, df_secondary], ignore_index=True)

# Aggregating spot counts across volunteers per group
group_counts = (
    df_clean
    .groupby(['day_id', 'group_id'])['spot_count']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
group_counts.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']

# Filling undefined std for single-volunteer groups
group_counts['std_count'] = group_counts['std_count'].fillna(0)

# Summing groups to get daily counts with propagated uncertainty
daily = (
    group_counts
    .groupby('day_id')
    .agg(
        daily_count=('mean_count', 'sum'),
        daily_uncertainty=('std_count', lambda x: np.sqrt((x**2).sum())),
        n_groups=('mean_count', 'count')
    )
    .reset_index()
)

# Flagging high-uncertainty days
daily['uncertainty_fraction'] = daily['daily_uncertainty'] / daily['daily_count'].replace(0, np.nan)
daily['quality_flag'] = daily['uncertainty_fraction'].apply(
    lambda x: 'high_uncertainty' if x > 0.5 else ('low_coverage' if pd.isna(x) else 'ok')
)

# Saving outputs
OUTPUT_DIR = 'Outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

daily.to_csv(os.path.join(OUTPUT_DIR, 'daily_sunspot_numbers.csv'), index=False)
group_counts.to_csv(os.path.join(OUTPUT_DIR, 'group_sunspot_numbers.csv'), index=False)
stats.to_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats.csv'), index=False)

# PIPELINE SUMMARY

# Volunteer quality control
print("VOLUNTEER QUALITY CONTROL")
print(stats.sort_values('relative_scatter', ascending=False).to_string(index=False))
print()
print(f"Accepted:                    {len(accepted_volunteers) - 1}")
print(f"Rejected (bad stats):        {len(stats) - (len(accepted_volunteers) - 1)}")
print(f"No teolixx overlap:          {len(no_overlap)}")
print(f"Total non-teolixx volunteers:{len(others['volunteer_id'].unique())}")
print(f"Secondary calibration: {len(secondary_accepted)} additional volunteers accepted")
print(f"Total accepted volunteers: {len(accepted_volunteers) + len(secondary_accepted)}")

# Coverage
print()
print("COVERAGE")
print(f"Total days:                  {daily['day_id'].nunique()}")
print(f"Total (day, group) pairs:    {len(group_counts)}")
print(f"Single-volunteer groups:     {(group_counts['n_volunteers'] == 1).sum()} / {len(group_counts)} ({100 * (group_counts['n_volunteers'] == 1).mean():.1f}%)")
print(f"Mean volunteers per group:   {group_counts['n_volunteers'].mean():.2f}")

# Daily count statistics
print()
print("DAILY COUNT STATISTICS")
print(daily['daily_count'].describe().round(2).to_string())
print(f"Zero-count days:             {(daily['daily_count'] == 0).sum()}")
print(f"High-uncertainty days:       {(daily['quality_flag'] == 'high_uncertainty').sum()} ({100 * (daily['quality_flag'] == 'high_uncertainty').mean():.1f}%)")

# Worst high-uncertainty days
print()
print("HIGH-UNCERTAINTY DAYS")
print(daily[daily['quality_flag'] == 'high_uncertainty']
      .sort_values('uncertainty_fraction', ascending=False)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'uncertainty_fraction']]
      .head(20).to_string(index=False))

# Top 10 highest count days
print()
print("TOP 10 HIGHEST COUNT DAYS")
print(daily.sort_values('daily_count', ascending=False)
      .head(10)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'quality_flag']]
      .to_string(index=False))

# Outputs saved
print()
print("OUTPUTS")
print("Saved: daily_sunspot_numbers.csv")
print("Saved: group_sunspot_numbers.csv")
print("Saved: volunteer_stats.csv")

# PLOT

daily_sorted = daily.sort_values('day_id').reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top panel: daily sunspot count with error bars
ax1.errorbar(
    range(len(daily_sorted)),
    daily_sorted['daily_count'],
    yerr=daily_sorted['daily_uncertainty'],
    fmt='o-', capsize=2, linewidth=0.8, markersize=2,
    alpha=0.6, label='Daily count'
)
flagged_idx = daily_sorted[daily_sorted['quality_flag'] == 'high_uncertainty'].index
ax1.scatter(
    flagged_idx,
    daily_sorted.loc[flagged_idx, 'daily_count'],
    color='red', zorder=5, s=10, label='High uncertainty'
)
ax1.set_ylabel('Daily sunspot count')
ax1.legend()

# Bottom panel: number of groups per day
ax2.bar(range(len(daily_sorted)), daily_sorted['n_groups'], color='steelblue', alpha=0.6)
ax2.set_ylabel('Number of groups')
ax2.set_xlabel('Day ID (index order)')

plt.suptitle('Daily sunspot numbers from citizen science data (full dataset)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'daily_sunspot_numbers_full.png'), dpi=150)
plt.show()


In [ ]:
# Libraries
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os

# Importing CSV dataset
cols_to_use = ['classification_id', 'user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv('sunspot-detectives-classifications.csv', usecols=cols_to_use)

# Removing rows where annotations and subject_data columns are empty
df = df.dropna(subset=['annotations', 'subject_data'])

# Reassigning database to the 4 columns needed
df = df[['user_name', 'user_ip', 'annotations', 'subject_data']]

# Extracting spot count from annotations column
def extract_count(annotations):
    try:
        result = json.loads(annotations)
        val = result[0]['value']
        
        # Structure 1: 'value' is a simple string or integer
        if isinstance(val, (int, float)):
            return int(val)
        if isinstance(val, str):
            val = val.strip() # Removes newlines
            if val == '': 
                return None   # Blank answers are dropped
            return int(val)
            
        # Structure 2: 'value' is a nested list
        if isinstance(val, list) and len(val) > 0:
            inner_val = val[0].get('value')
            if isinstance(inner_val, int):
                return inner_val
            label = val[0].get('label')
            if label is not None:
                return int(label)
                
    except (ValueError, KeyError, TypeError, IndexError, json.JSONDecodeError):
        return None

# Storing spot count data in new column
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)

# Extracting subject data column
def extract_filename(subject_data):
    try:
        result = json.loads(subject_data)
        first_val = list(result.values())[0]
        filename = first_val['Filename']
        name = filename.replace('.png', '')
        parts = name.split('_')
        return (parts[0], parts[1])
    except (ValueError, KeyError, TypeError, json.JSONDecodeError, IndexError):
        return (None, None)

# Storing Day ID and Group ID data in 2 new columns
df[['day_id', 'group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)

# Extracting volunteer identity
def assign_volunteer_id(row):
    user_name = row['user_name']
    user_ip = row['user_ip']
    if pd.notna(user_name):
        return user_name
    if pd.notna(user_ip):
        return 'anon_' + str(user_ip)
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

# Dropping now-unnecessary columns
df = df.drop(columns=['annotations', 'subject_data', 'user_name', 'user_ip'])

# Separating teolixx from all others
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

# Resolving teolixx duplicates by taking median
teolixx = teolixx.groupby(['day_id', 'group_id'])['spot_count'].median().reset_index()
teolixx.columns = ['day_id', 'group_id', 'spot_count']

# Merging other volunteers' counts with teolixx's counts on same images
merged = pd.merge(
    others,
    teolixx[['day_id', 'group_id', 'spot_count']],
    on=['day_id', 'group_id'],
    suffixes=('_volunteer', '_teolixx')
)

# Difference between volunteer's count and teolixx's count
merged['diff'] = merged['spot_count_volunteer'] - merged['spot_count_teolixx']

# Computing per-volunteer bias, scatter, and number of overlapping images
stats = merged.groupby('volunteer_id')['diff'].agg(['mean', 'std', 'count']).reset_index()
stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap']

# Filling NaN scatter values
stats['scatter'] = stats['scatter'].fillna(0)

# Computing mean teolixx count on each volunteer's overlapping images
teolixx_mean_on_overlap = (
    merged.groupby('volunteer_id')['spot_count_teolixx']
    .mean()
    .reset_index()
)
teolixx_mean_on_overlap.columns = ['volunteer_id', 'teolixx_mean_count']

# Merging into stats
stats = stats.merge(teolixx_mean_on_overlap, on='volunteer_id')

# Computing relative scatter: scatter as fraction of teolixx's mean count on shared images
stats['relative_scatter'] = stats['scatter'] / stats['teolixx_mean_count']

# Setting thresholds: can be adjusted
MIN_OVERLAP = 5             # Must have counted at least this many images teolixx also counted
MAX_RELATIVE_SCATTER = 0.5  # Scatter must be below 50% of teolixx's mean count on shared images

# Considering volunteers in range of thresholds
accepted_volunteers = stats[
    (stats['n_overlap'] >= MIN_OVERLAP) &
    (stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
]['volunteer_id'].tolist()

# Always including teolixx
accepted_volunteers.append('teolixx')

# Volunteers with no overlap with teolixx
no_overlap = set(others['volunteer_id'].unique()) - set(stats['volunteer_id'].unique())

# Filtering data to accepted volunteers only
df_clean = df[df['volunteer_id'].isin(accepted_volunteers)].copy()

# Collapsing within-volunteer duplicates by averaging
df_clean = (
    df_clean
    .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
    .mean()
    .reset_index()
)

# SECONDARY CALIBRATION: no-overlap volunteers vs accepted ensemble

# Build per-group reference mean from accepted volunteers (excluding teolixx)
accepted_no_teolixx = df_clean[df_clean['volunteer_id'] != 'teolixx']
accepted_group_means = (
    accepted_no_teolixx
    .groupby(['day_id', 'group_id'])['spot_count']
    .median()
    .reset_index()
    .rename(columns={'spot_count': 'accepted_mean'})
)

# Raw counts for no-overlap volunteers
no_overlap_df = df[df['volunteer_id'].isin(no_overlap)].copy()

# Merge on shared (day_id, group_id) pairs
merged_no_overlap = pd.merge(
    no_overlap_df,
    accepted_group_means,
    on=['day_id', 'group_id']
)

secondary_accepted = []

if len(merged_no_overlap) > 0:
    merged_no_overlap['diff'] = merged_no_overlap['spot_count'] - merged_no_overlap['accepted_mean']

    no_overlap_stats = (
        merged_no_overlap
        .groupby('volunteer_id')['diff']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    no_overlap_stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap_accepted']
    no_overlap_stats['scatter'] = no_overlap_stats['scatter'].fillna(0)

    accepted_mean_on_overlap = (
        merged_no_overlap
        .groupby('volunteer_id')['accepted_mean']
        .mean()
        .reset_index()
        .rename(columns={'accepted_mean': 'accepted_mean_count'})
    )
    no_overlap_stats = no_overlap_stats.merge(accepted_mean_on_overlap, on='volunteer_id')
    no_overlap_stats['relative_scatter'] = (
        no_overlap_stats['scatter'] / no_overlap_stats['accepted_mean_count']
    )

    secondary_accepted = no_overlap_stats[
        (no_overlap_stats['n_overlap_accepted'] >= MIN_OVERLAP) &
        (no_overlap_stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
    ]['volunteer_id'].tolist()

    # Add secondary-accepted volunteers into df_clean
    df_secondary = df[df['volunteer_id'].isin(secondary_accepted)].copy()
    df_secondary = (
        df_secondary
        .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
        .mean()
        .reset_index()
    )
    df_clean = pd.concat([df_clean, df_secondary], ignore_index=True)

# Volunteers excluded from both primary and secondary calibration
rejected_volunteers = set(stats[
    ~(
        (stats['n_overlap'] >= MIN_OVERLAP) &
        (stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
    )
]['volunteer_id'].tolist())

truly_excluded = (no_overlap - set(secondary_accepted)) | rejected_volunteers

# Include their raw counts
df_excluded = df[df['volunteer_id'].isin(truly_excluded)].copy()
df_excluded = (
    df_excluded
    .groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count']
    .mean()
    .reset_index()
)

# Combine everything before outlier detection
df_all = pd.concat([df_clean, df_excluded], ignore_index=True)

# Now run outlier detection on the full combined set
group_medians = df_all.groupby(['day_id', 'group_id'])['spot_count'].median().reset_index()
group_medians.columns = ['day_id', 'group_id', 'group_median']
df_all = pd.merge(df_all, group_medians, on=['day_id', 'group_id'])

df_all['is_outlier'] = (
    np.abs(df_all['spot_count'] - df_all['group_median']) > 
    np.maximum(15, 0.5 * df_all['group_median'])
)

df_clean_final = df_all[~df_all['is_outlier']].drop(columns=['group_median', 'is_outlier'])

# Aggregating spot counts across volunteers per group
group_counts = (
    df_clean_final
    .groupby(['day_id', 'group_id'])['spot_count']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
group_counts.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']

# Filling undefined std for single-volunteer groups
group_counts['std_count'] = group_counts['std_count'].fillna(0)

# Summing groups to get daily counts with propagated uncertainty
daily = (
    group_counts
    .groupby('day_id')
    .agg(
        daily_count=('mean_count', 'sum'),
        daily_uncertainty=('std_count', lambda x: np.sqrt((x**2).sum())),
        n_groups=('mean_count', 'count')
    )
    .reset_index()
)

# Flagging high-uncertainty days
daily['uncertainty_fraction'] = daily['daily_uncertainty'] / daily['daily_count'].replace(0, np.nan)
daily['quality_flag'] = daily['uncertainty_fraction'].apply(
    lambda x: 'high_uncertainty' if x > 0.5 else ('low_coverage' if pd.isna(x) else 'ok')
)

# Saving outputs
OUTPUT_DIR = 'Outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if len(no_overlap_stats) > 0:
    all_volunteer_stats = pd.concat([stats, no_overlap_stats], ignore_index=True)
else:
    all_volunteer_stats = stats

daily.to_csv(os.path.join(OUTPUT_DIR, 'daily_sunspot_numbers.csv'), index=False)
group_counts.to_csv(os.path.join(OUTPUT_DIR, 'group_sunspot_numbers.csv'), index=False)
all_volunteer_stats.to_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats.csv'), index=False)

# PIPELINE SUMMARY

# Volunteer quality control
print("VOLUNTEER QUALITY CONTROL")
print(stats.sort_values('relative_scatter', ascending=False).to_string(index=False))
print()
print(f"Accepted:                    {len(accepted_volunteers) - 1}")
print(f"Rejected (bad stats):        {len(stats) - (len(accepted_volunteers) - 1)}")
print(f"No teolixx overlap:          {len(no_overlap)}")
print(f"Total non-teolixx volunteers:{len(others['volunteer_id'].unique())}")
print(f"Secondary calibration: {len(secondary_accepted)} additional volunteers accepted")
print(f"Total accepted volunteers: {len(accepted_volunteers) + len(secondary_accepted)}")

# Coverage
print()
print("COVERAGE")
print(f"Total days:                  {daily['day_id'].nunique()}")
print(f"Total (day, group) pairs:    {len(group_counts)}")
print(f"Single-volunteer groups:     {(group_counts['n_volunteers'] == 1).sum()} / {len(group_counts)} ({100 * (group_counts['n_volunteers'] == 1).mean():.1f}%)")
print(f"Mean volunteers per group:   {group_counts['n_volunteers'].mean():.2f}")

# Daily count statistics
print()
print("DAILY COUNT STATISTICS")
print(daily['daily_count'].describe().round(2).to_string())
print(f"Zero-count days:             {(daily['daily_count'] == 0).sum()}")
print(f"High-uncertainty days:       {(daily['quality_flag'] == 'high_uncertainty').sum()} ({100 * (daily['quality_flag'] == 'high_uncertainty').mean():.1f}%)")

# Worst high-uncertainty days
print()
print("HIGH-UNCERTAINTY DAYS")
print(daily[daily['quality_flag'] == 'high_uncertainty']
      .sort_values('uncertainty_fraction', ascending=False)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'uncertainty_fraction']]
      .head(20).to_string(index=False))

# Top 10 highest count days
print()
print("TOP 10 HIGHEST COUNT DAYS")
print(daily.sort_values('daily_count', ascending=False)
      .head(10)
      [['day_id', 'daily_count', 'daily_uncertainty', 'n_groups', 'quality_flag']]
      .to_string(index=False))

group_counts_check = pd.read_csv('Outputs/group_sunspot_numbers.csv')
print(f"Sum of mean_count:   {group_counts_check['mean_count'].sum():.0f}")
print(f"Sum of n_volunteers: {group_counts_check['n_volunteers'].sum():.0f}")
print(f"Rows before outlier filter: {len(df_all)}")
print(f"Rows removed as outliers:   {df_all['is_outlier'].sum()}")
print(f"Rows after outlier filter:  {len(df_clean_final)}")

# Outputs saved
print()
print("OUTPUTS")
print("Saved: daily_sunspot_numbers.csv")
print("Saved: group_sunspot_numbers.csv")
print("Saved: volunteer_stats.csv")

# PLOT

daily_sorted = daily.sort_values('day_id').reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top panel: daily sunspot count with error bars
ax1.errorbar(
    range(len(daily_sorted)),
    daily_sorted['daily_count'],
    yerr=daily_sorted['daily_uncertainty'],
    fmt='o-', capsize=2, linewidth=0.8, markersize=2,
    alpha=0.6, label='Daily count'
)
flagged_idx = daily_sorted[daily_sorted['quality_flag'] == 'high_uncertainty'].index
ax1.scatter(
    flagged_idx,
    daily_sorted.loc[flagged_idx, 'daily_count'],
    color='red', zorder=5, s=10, label='High uncertainty'
)
ax1.set_ylabel('Daily sunspot count')
ax1.legend()

# Bottom panel: number of groups per day
ax2.bar(range(len(daily_sorted)), daily_sorted['n_groups'], color='steelblue', alpha=0.6)
ax2.set_ylabel('Number of groups')
ax2.set_xlabel('Day ID (index order)')

plt.suptitle('Daily sunspot numbers from citizen science data (full dataset)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'daily_sunspot_numbers_full.png'), dpi=150)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os
from scipy.stats import norm

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_PATH   = 'sunspot-detectives-classifications.csv'
OUTPUT_DIR  = 'Outputs/Final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_OVERLAP          = 5     # minimum images shared with teolixx
MAX_RELATIVE_SCATTER = 0.5   # maximum scatter relative to teolixx mean
BIAS_RELIABLE_N      = 20    # minimum overlap to trust bias estimate
Z_OUTLIER_THRESHOLD  = 2.5   # z-score cutoff for within-group outliers

# ── PARSE ────────────────────────────────────────────────────────────────────
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str):
            val = val.strip()
            return int(val) if val else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['spot_count']   = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count']   = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']
df = df.drop(columns=['annotations','subject_data','user_name','user_ip'])

print(f"Rows after parsing: {len(df):,}")

# ── TEOLIXX REFERENCE ────────────────────────────────────────────────────────
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others  = df[df['volunteer_id'] != 'teolixx'].copy()

teolixx_ref = (teolixx.groupby(['day_id','group_id'])['spot_count']
               .median().reset_index()
               .rename(columns={'spot_count':'teolixx_count'}))

# ── PRIMARY CALIBRATION ──────────────────────────────────────────────────────
merged = pd.merge(others, teolixx_ref, on=['day_id','group_id'])
merged['diff'] = merged['spot_count'] - merged['teolixx_count']

stats = (merged.groupby('volunteer_id')['diff']
         .agg(['mean','std','count']).reset_index()
         .rename(columns={'mean':'bias','std':'scatter','count':'n_overlap'}))
stats['scatter'] = stats['scatter'].fillna(0)

teolixx_mean = (merged.groupby('volunteer_id')['teolixx_count']
                .mean().reset_index()
                .rename(columns={'teolixx_count':'teolixx_mean_count'}))
stats = stats.merge(teolixx_mean, on='volunteer_id')
stats['relative_scatter'] = stats['scatter'] / stats['teolixx_mean_count']

accepted_mask = (stats['n_overlap'] >= MIN_OVERLAP) & (stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
accepted_stats = stats[accepted_mask].copy()
accepted_ids   = accepted_stats['volunteer_id'].tolist() + ['teolixx']

print(f"Primary accepted: {len(accepted_stats)}")

# ── BIAS CORRECTION (reliable volunteers only) ────────────────────────────────
reliable_bias = accepted_stats[accepted_stats['n_overlap'] >= BIAS_RELIABLE_N]
bias_map = dict(zip(reliable_bias['volunteer_id'], reliable_bias['bias']))
bias_map['teolixx'] = 0.0
print(f"Volunteers with reliable bias estimate (n_overlap >= {BIAS_RELIABLE_N}): {len(reliable_bias)}")

# ── BUILD WORKING DATASET ────────────────────────────────────────────────────
df_work = df[df['volunteer_id'].isin(accepted_ids)].copy()
df_work['bias_correction'] = df_work['volunteer_id'].map(bias_map).fillna(0.0)
df_work['spot_count'] = (df_work['spot_count'] - df_work['bias_correction']).clip(lower=0)
df_work = (df_work.groupby(['volunteer_id','day_id','group_id'])['spot_count']
           .mean().reset_index())

# ── Z-SCORE OUTLIER REMOVAL ──────────────────────────────────────────────────
group_stats = (df_work.groupby(['day_id','group_id'])['spot_count']
               .agg(['mean','std','count']).reset_index()
               .rename(columns={'mean':'group_mean','std':'group_std','count':'n_vol_pre'}))
group_stats['group_std'] = group_stats['group_std'].fillna(0)

df_work = df_work.merge(group_stats[['day_id','group_id','group_mean','group_std']], on=['day_id','group_id'])
df_work['z_score'] = np.where(
    df_work['group_std'] > 0,
    (df_work['spot_count'] - df_work['group_mean']) / df_work['group_std'],
    0.0
)
df_work['is_outlier'] = df_work['z_score'].abs() > Z_OUTLIER_THRESHOLD
n_outliers = df_work['is_outlier'].sum()
print(f"Outliers removed (|z| > {Z_OUTLIER_THRESHOLD}): {n_outliers} ({100*n_outliers/len(df_work):.2f}%)")

df_clean = df_work[~df_work['is_outlier']].drop(columns=['group_mean','group_std','z_score','is_outlier'])

# ── AGGREGATION ──────────────────────────────────────────────────────────────
group_counts = (df_clean.groupby(['day_id','group_id'])['spot_count']
                .agg(['mean','std','count']).reset_index()
                .rename(columns={'mean':'mean_count','std':'std_count','count':'n_volunteers'}))
group_counts['std_count'] = group_counts['std_count'].fillna(0)

daily = (group_counts.groupby('day_id')
         .agg(
             daily_count       = ('mean_count', 'sum'),
             daily_uncertainty = ('std_count',  lambda x: np.sqrt((x**2).sum())),
             n_groups          = ('mean_count', 'count')
         ).reset_index())
daily['uncertainty_fraction'] = daily['daily_uncertainty'] / daily['daily_count'].replace(0, np.nan)
daily['quality_flag'] = daily['uncertainty_fraction'].apply(
    lambda x: 'high_uncertainty' if x > 0.5 else ('low_coverage' if pd.isna(x) else 'ok'))

# ── SAVE OUTPUTS ─────────────────────────────────────────────────────────────
daily.to_csv(os.path.join(OUTPUT_DIR, 'daily_sunspot_numbers_final.csv'), index=False)
group_counts.to_csv(os.path.join(OUTPUT_DIR, 'group_sunspot_numbers_final.csv'), index=False)
accepted_stats.to_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats_final.csv'), index=False)
print("CSVs saved.")

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print(f"\nFINAL PIPELINE SUMMARY")
print(f"Rows parsed:              {len(df):,}")
print(f"Accepted volunteers:      {len(accepted_stats)}")
print(f"  of which bias-corrected:{len(reliable_bias)}")
print(f"Outlier classifications:  {n_outliers}")
print(f"Total days:               {daily['day_id'].nunique():,}")
print(f"Total group pairs:        {len(group_counts):,}")
print(f"Mean volunteers/group:    {group_counts['n_volunteers'].mean():.2f}")
print(f"Daily count mean:         {daily['daily_count'].mean():.2f}")
print(f"Daily count max:          {daily['daily_count'].max():.2f}")
print(f"High-uncertainty days:    {(daily['quality_flag']=='high_uncertainty').sum()} ({100*(daily['quality_flag']=='high_uncertainty').mean():.1f}%)")

# ── FINAL FIGURE: Daily sunspot time series ───────────────────────────────────
daily_sorted = daily.sort_values('day_id').reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

flagged = daily_sorted[daily_sorted['quality_flag'] == 'high_uncertainty'].index

ax1.errorbar(range(len(daily_sorted)), daily_sorted['daily_count'],
             yerr=daily_sorted['daily_uncertainty'],
             fmt='o-', capsize=2, linewidth=0.8, markersize=2, alpha=0.6, color='steelblue', label='Daily count')
ax1.scatter(flagged, daily_sorted.loc[flagged, 'daily_count'],
            color='red', zorder=5, s=10, label='High uncertainty')
ax1.set_ylabel('Daily sunspot count')
ax1.legend()
ax1.set_title('Final calibrated daily sunspot numbers\n'
              f'(n_overlap ≥ {MIN_OVERLAP}, rel_scatter ≤ {MAX_RELATIVE_SCATTER}, '
              f'bias correction for n_overlap ≥ {BIAS_RELIABLE_N}, z-outlier threshold = {Z_OUTLIER_THRESHOLD})')

ax2.bar(range(len(daily_sorted)), daily_sorted['n_groups'], color='steelblue', alpha=0.6)
ax2.set_ylabel('Groups per session')
ax2.set_xlabel('Observing session (sorted by day_id)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_daily_sunspot_numbers.png'), dpi=150)
plt.show()
print("Saved: final_daily_sunspot_numbers.png")


## Initial Analyses & Changes

Scripts documenting the early analysis logic and structural changes.

In [ ]:
# Each of these analyses attempts to decide what parameters to choose.
# All 5 analyses have been included in this file, since it is meant for reference, not instruction. Search (Cmd + F) for Analysis <Number> for any specific one.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

OUTPUT_DIR = 'Outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the saved volunteer stats
stats = pd.read_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats.csv'))

# Filter to only primary calibration volunteers (those with teolixx_mean_count column populated)
primary = stats.dropna(subset=['teolixx_mean_count']).copy()
primary = primary[primary['n_overlap'].notna()].copy()

MIN_OVERLAP = 5
MAX_RELATIVE_SCATTER = 0.5

primary['accepted'] = (
    (primary['n_overlap'] >= MIN_OVERLAP) &
    (primary['relative_scatter'] <= MAX_RELATIVE_SCATTER)
)

# ANALYSIS 1: Volunteer quality space

fig, ax = plt.subplots(figsize=(10, 7))

colors = primary['accepted'].map({True: '#2ecc71', False: '#e74c3c'})
ax.scatter(
    primary['n_overlap'],
    primary['relative_scatter'],
    c=colors, alpha=0.4, s=15, linewidths=0
)

ax.axvline(x=MIN_OVERLAP, color='black', linewidth=1.5, linestyle='--', label=f'Min overlap = {MIN_OVERLAP}')
ax.axhline(y=MAX_RELATIVE_SCATTER, color='black', linewidth=1.5, linestyle=':', label=f'Max relative scatter = {MAX_RELATIVE_SCATTER}')

ax.set_xlabel('Number of images shared with teolixx (n_overlap)', fontsize=12)
ax.set_ylabel('Relative scatter (scatter / teolixx mean count)', fontsize=12)
ax.set_title('Volunteer quality space: primary calibration', fontsize=13)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(left=1)
ax.set_ylim(bottom=0.001)

n_accepted = primary['accepted'].sum()
n_total = len(primary)
green_patch = mpatches.Patch(color='#2ecc71', label=f'Accepted ({n_accepted})')
red_patch = mpatches.Patch(color='#e74c3c', label=f'Rejected ({n_total - n_accepted})')
ax.legend(handles=[green_patch, red_patch] + ax.get_legend_handles_labels()[0][-2:], fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_volunteer_quality_space.png'), dpi=150)
plt.show()
print("Saved: plot_volunteer_quality_space.png")

# ANALYSIS 2: Parameter sensitivity sweep

cols_to_use = ['classification_id', 'user_name', 'user_ip', 'annotations', 'subject_data']
import json

df_raw = pd.read_csv(
    'sunspot-detectives-classifications.csv',
    usecols=cols_to_use
)
df_raw = df_raw.dropna(subset=['annotations', 'subject_data'])

def extract_count(annotations):
    try:
        result = json.loads(annotations)
        val = result[0]['value']
        if isinstance(val, (int, float)):
            return int(val)
        if isinstance(val, str):
            val = val.strip()
            if val == '':
                return None
            return int(val)
        if isinstance(val, list) and len(val) > 0:
            inner_val = val[0].get('value')
            if isinstance(inner_val, int):
                return inner_val
            label = val[0].get('label')
            if label is not None:
                return int(label)
    except:
        return None

def extract_filename(subject_data):
    try:
        result = json.loads(subject_data)
        first_val = list(result.values())[0]
        filename = first_val['Filename']
        name = filename.replace('.png', '')
        parts = name.split('_')
        return (parts[0], parts[1])
    except:
        return (None, None)

def assign_volunteer_id(row):
    if pd.notna(row['user_name']):
        return row['user_name']
    if pd.notna(row['user_ip']):
        return 'anon_' + str(row['user_ip'])
    return 'unknown'

df_raw['spot_count'] = df_raw['annotations'].apply(extract_count)
df_raw = df_raw.dropna(subset=['spot_count'])
df_raw['spot_count'] = df_raw['spot_count'].astype(int)
df_raw[['day_id', 'group_id']] = df_raw['subject_data'].apply(extract_filename).apply(pd.Series)
df_raw['volunteer_id'] = df_raw[['user_name', 'user_ip']].apply(assign_volunteer_id, axis=1)
df_raw = df_raw[df_raw['volunteer_id'] != 'unknown']
df_raw = df_raw.drop(columns=['annotations', 'subject_data', 'user_name', 'user_ip'])

teolixx = df_raw[df_raw['volunteer_id'] == 'teolixx'].copy()
others = df_raw[df_raw['volunteer_id'] != 'teolixx'].copy()
teolixx = teolixx.groupby(['day_id', 'group_id'])['spot_count'].median().reset_index()
teolixx.columns = ['day_id', 'group_id', 'spot_count']

merged = pd.merge(others, teolixx[['day_id', 'group_id', 'spot_count']], on=['day_id', 'group_id'], suffixes=('_volunteer', '_teolixx'))
merged['diff'] = merged['spot_count_volunteer'] - merged['spot_count_teolixx']

base_stats = merged.groupby('volunteer_id')['diff'].agg(['mean', 'std', 'count']).reset_index()
base_stats.columns = ['volunteer_id', 'bias', 'scatter', 'n_overlap']
base_stats['scatter'] = base_stats['scatter'].fillna(0)
teolixx_mean_on_overlap = merged.groupby('volunteer_id')['spot_count_teolixx'].mean().reset_index()
teolixx_mean_on_overlap.columns = ['volunteer_id', 'teolixx_mean_count']
base_stats = base_stats.merge(teolixx_mean_on_overlap, on='volunteer_id')
base_stats['relative_scatter'] = base_stats['scatter'] / base_stats['teolixx_mean_count']

# Sweep parameters
min_overlaps = [3, 5, 10, 20]
max_scatters = [0.2, 0.3, 0.5, 0.7, 1.0]

results = []

for mo in min_overlaps:
    for ms in max_scatters:
        accepted = base_stats[
            (base_stats['n_overlap'] >= mo) &
            (base_stats['relative_scatter'] <= ms)
        ]['volunteer_id'].tolist()
        accepted.append('teolixx')

        df_accepted = df_raw[df_raw['volunteer_id'].isin(accepted)].copy()
        df_accepted = df_accepted.groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count'].mean().reset_index()

        group_counts = df_accepted.groupby(['day_id', 'group_id'])['spot_count'].agg(['mean', 'std', 'count']).reset_index()
        group_counts.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']
        group_counts['std_count'] = group_counts['std_count'].fillna(0)

        multi_vol = group_counts[group_counts['n_volunteers'] >= 2]
        mean_within_std = multi_vol['std_count'].mean() if len(multi_vol) > 0 else np.nan
        single_vol_frac = (group_counts['n_volunteers'] == 1).mean()

        results.append({
            'min_overlap': mo,
            'max_relative_scatter': ms,
            'n_accepted': len(accepted) - 1,
            'mean_within_group_std': round(mean_within_std, 3),
            'single_volunteer_fraction': round(single_vol_frac, 3)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUTPUT_DIR, 'parameter_sensitivity.csv'), index=False)

# Plot parameter sensitivity as heatmaps

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(
    axes,
    ['n_accepted', 'mean_within_group_std', 'single_volunteer_fraction'],
    ['Accepted volunteers', 'Mean within-group std', 'Fraction of single-volunteer groups']
):
    pivot = results_df.pivot(index='max_relative_scatter', columns='min_overlap', values=col)
    im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(min_overlaps)))
    ax.set_xticklabels(min_overlaps)
    ax.set_yticks(range(len(max_scatters)))
    ax.set_yticklabels(max_scatters)
    ax.set_xlabel('Min overlap')
    ax.set_ylabel('Max relative scatter')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

    xi = min_overlaps.index(5)
    yi = max_scatters.index(0.5)
    ax.add_patch(plt.Rectangle((xi - 0.5, yi - 0.5), 1, 1, fill=False, edgecolor='red', linewidth=2))

plt.suptitle('Parameter sensitivity (red box = current settings)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_parameter_sensitivity.png'), dpi=150)
plt.show()
print("Saved: plot_parameter_sensitivity.png")

# ANALYSIS 3: Within-group spread (primary accepted only, current params)

accepted_ids = base_stats[
    (base_stats['n_overlap'] >= MIN_OVERLAP) &
    (base_stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
]['volunteer_id'].tolist()
accepted_ids.append('teolixx')

df_accepted_only = df_raw[df_raw['volunteer_id'].isin(accepted_ids)].copy()
df_accepted_only = df_accepted_only.groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count'].mean().reset_index()

group_accepted = df_accepted_only.groupby(['day_id', 'group_id'])['spot_count'].agg(['mean', 'std', 'count']).reset_index()
group_accepted.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']
group_accepted['std_count'] = group_accepted['std_count'].fillna(0)
group_accepted['relative_std'] = group_accepted['std_count'] / group_accepted['mean_count'].replace(0, np.nan)

multi = group_accepted[group_accepted['n_volunteers'] >= 2].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(multi['std_count'].clip(upper=20), bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('Within-group std (spot counts)', fontsize=11)
axes[0].set_ylabel('Number of groups')
axes[0].set_title(f'Within-group disagreement (primary accepted, n={len(multi):,} groups)')
axes[0].axvline(multi['std_count'].median(), color='red', linestyle='--', label=f'Median = {multi["std_count"].median():.2f}')
axes[0].legend()

axes[1].scatter(multi['mean_count'], multi['std_count'], alpha=0.15, s=4, color='steelblue')
axes[1].set_xlabel('Group mean count', fontsize=11)
axes[1].set_ylabel('Within-group std', fontsize=11)
axes[1].set_title('Within-group std vs mean count')
axes[1].set_xlim(0, 60)
axes[1].set_ylim(0, 20)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_within_group_spread.png'), dpi=150)
plt.show()
print("Saved: plot_within_group_spread.png")

# ANALYSIS 4: Bias distribution of accepted volunteers

accepted_stats = base_stats[
    (base_stats['n_overlap'] >= MIN_OVERLAP) &
    (base_stats['relative_scatter'] <= MAX_RELATIVE_SCATTER)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(accepted_stats['bias'].clip(-20, 20), bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='black', linewidth=1.5, linestyle='-', label='Zero bias')
axes[0].axvline(accepted_stats['bias'].mean(), color='red', linestyle='--', label=f'Mean bias = {accepted_stats["bias"].mean():.2f}')
axes[0].set_xlabel('Bias (volunteer mean − teolixx mean, in spots)', fontsize=11)
axes[0].set_ylabel('Number of volunteers')
axes[0].set_title('Systematic bias of accepted volunteers')
axes[0].legend()

axes[1].scatter(accepted_stats['n_overlap'], accepted_stats['bias'], alpha=0.4, s=15, color='steelblue')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xscale('log')
axes[1].set_xlabel('n_overlap (log scale)')
axes[1].set_ylabel('Bias')
axes[1].set_title('Bias vs number of shared images')
axes[1].set_ylim(-30, 30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_bias_distribution.png'), dpi=150)
plt.show()
print("Saved: plot_bias_distribution.png")

# ANALYSIS 5: Daily count under different parameter choices

def daily_counts_for_params(df, base_stats_df, min_overlap, max_scatter):
    acc = base_stats_df[
        (base_stats_df['n_overlap'] >= min_overlap) &
        (base_stats_df['relative_scatter'] <= max_scatter)
    ]['volunteer_id'].tolist()
    acc.append('teolixx')

    df_a = df[df['volunteer_id'].isin(acc)].copy()
    df_a = df_a.groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count'].mean().reset_index()
    gc = df_a.groupby(['day_id', 'group_id'])['spot_count'].agg(['mean', 'std']).reset_index()
    gc.columns = ['day_id', 'group_id', 'mean_count', 'std_count']
    gc['std_count'] = gc['std_count'].fillna(0)
    daily = gc.groupby('day_id').agg(daily_count=('mean_count', 'sum')).reset_index()
    return daily.sort_values('day_id').reset_index(drop=True)

param_sets = [
    (5, 0.3, 'Strict (n≥5, rs≤0.3)'),
    (5, 0.5, 'Current (n≥5, rs≤0.5)'),
    (5, 0.7, 'Loose (n≥5, rs≤0.7)'),
    (10, 0.5, 'High overlap (n≥10, rs≤0.5)')
]

fig, ax = plt.subplots(figsize=(16, 5))

for min_o, max_s, label in param_sets:
    d = daily_counts_for_params(df_raw, base_stats, min_o, max_s)
    ax.plot(range(len(d)), d['daily_count'], linewidth=0.8, alpha=0.8, label=label)

ax.set_xlabel('Observing session (sorted by day_id)')
ax.set_ylabel('Daily sunspot count')
ax.set_title('Daily count stability under different calibration parameters')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_daily_count_stability.png'), dpi=150)
plt.show()
print("Saved: plot_daily_count_stability.png")


In [ ]:
# All 3 changes have been included in this file, since it is meant for reference, not instruction. Search (Cmd + F) for Change <Number> for any specific one.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = 'Outputs'
stats = pd.read_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats.csv'))

# CHANGE 1: Apply bias correction only to well-measured volunteers

RELIABLE_BIAS_OVERLAP = 20

def get_daily_counts_v2(df, accepted_ids, accepted_stats_df, bias_overlap_threshold):
    df_a = df[df['volunteer_id'].isin(accepted_ids)].copy()
    
    reliable = accepted_stats_df[accepted_stats_df['n_overlap'] >= bias_overlap_threshold]
    bias_map = dict(zip(reliable['volunteer_id'], reliable['bias']))
    bias_map['teolixx'] = 0.0
    
    df_a['bias_correction'] = df_a['volunteer_id'].map(bias_map).fillna(0.0)
    df_a['spot_count_corrected'] = (df_a['spot_count'] - df_a['bias_correction']).clip(lower=0)
    
    df_a = df_a.groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count_corrected'].mean().reset_index()
    df_a.columns = ['volunteer_id', 'day_id', 'group_id', 'spot_count']
    
    gc = df_a.groupby(['day_id', 'group_id'])['spot_count'].agg(['mean', 'std', 'count']).reset_index()
    gc.columns = ['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']
    gc['std_count'] = gc['std_count'].fillna(0)
    
    daily = gc.groupby('day_id').agg(
        daily_count=('mean_count', 'sum'),
        daily_uncertainty=('std_count', lambda x: np.sqrt((x**2).sum())),
        n_groups=('mean_count', 'count')
    ).reset_index().sort_values('day_id').reset_index(drop=True)
    
    return daily, gc

daily_none, gc_none = get_daily_counts_v2(df_raw, accepted_ids, accepted_stats, bias_overlap_threshold=99999)
daily_all, gc_all = get_daily_counts_v2(df_raw, accepted_ids, accepted_stats, bias_overlap_threshold=0)
daily_reliable, gc_reliable = get_daily_counts_v2(df_raw, accepted_ids, accepted_stats, bias_overlap_threshold=20)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

idx = range(len(daily_none))
axes[0].plot(idx, daily_none['daily_count'], color='steelblue', linewidth=0.6, alpha=0.7, label='No correction')
axes[0].plot(idx, daily_reliable['daily_count'], color='tomato', linewidth=0.6, alpha=0.7, label='Reliable-only correction (n_overlap ≥ 20)')
axes[0].plot(idx, daily_all['daily_count'], color='gray', linewidth=0.6, alpha=0.5, label='All corrected (n_overlap ≥ 5)')
axes[0].set_ylabel('Daily sunspot count')
axes[0].set_title('Effect of bias correction threshold on daily count')
axes[0].legend()

diff_reliable = daily_reliable['daily_count'] - daily_none['daily_count']
diff_all = daily_all['daily_count'] - daily_none['daily_count']

axes[1].plot(idx, diff_reliable, color='tomato', linewidth=0.6, alpha=0.8, label=f'Reliable-only: mean={diff_reliable.mean():.2f}')
axes[1].plot(idx, diff_all, color='gray', linewidth=0.6, alpha=0.6, label=f'All corrected: mean={diff_all.mean():.2f}')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('Difference (corrected − uncorrected)')
axes[1].set_xlabel('Observing session (sorted by day_id)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_bias_correction_threshold.png'), dpi=150)
plt.show()

for gc, label in [(gc_none, 'No correction'), (gc_reliable, 'Reliable-only'), (gc_all, 'All corrected')]:
    multi = gc[gc['n_volunteers'] >= 2].copy()
    multi['rel_std'] = multi['std_count'] / multi['mean_count'].replace(0, np.nan)
    print(f"{label}: median rel std = {multi['rel_std'].median():.3f}, median abs std = {multi['std_count'].median():.3f}")

# CHANGE 2: Group-level stability

gc_work = gc_none.copy()
gc_work['rel_std'] = gc_work['std_count'] / gc_work['mean_count'].replace(0, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

multi = gc_work[gc_work['n_volunteers'] >= 2].copy()
bins = [2, 3, 4, 5, 7, 10, 15, 20, 50]
labels_b = ['2', '3', '4', '5-6', '7-9', '10-14', '15-19', '20+']
multi['n_bin'] = pd.cut(multi['n_volunteers'], bins=bins, labels=labels_b)

bp_data = [multi[multi['n_bin'] == lbl]['rel_std'].dropna().values for lbl in labels_b]
bp = axes[0].boxplot(bp_data, labels=labels_b, showfliers=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
axes[0].set_xlabel('Number of volunteers per group')
axes[0].set_ylabel('Relative within-group std (std / mean)')
axes[0].set_title('Spread decreases with more observers\n(as expected)')
axes[0].set_ylim(0, 1)

axes[1].scatter(gc_work['mean_count'], gc_work['rel_std'], alpha=0.1, s=4, color='steelblue')
axes[1].set_xlabel('Group mean count (sunspots)')
axes[1].set_ylabel('Relative within-group std')
axes[1].set_title('Spread vs activity level')
axes[1].set_xlim(0, 60)
axes[1].set_ylim(0, 2)
count_bins = np.arange(0, 61, 5)
for i in range(len(count_bins)-1):
    mask = (gc_work['mean_count'] >= count_bins[i]) & (gc_work['mean_count'] < count_bins[i+1])
    if mask.sum() >= 10:
        med = gc_work.loc[mask, 'rel_std'].median()
        axes[1].plot(count_bins[i] + 2.5, med, 'ro', markersize=5)

thresholds = [0.1, 0.2, 0.3, 0.5]
colors_t = ['#1a237e', '#1565c0', '#42a5f5', '#b3e5fc']
n_vol_range = range(2, 21)
for thresh, col in zip(thresholds, colors_t):
    fracs = []
    for nv in n_vol_range:
        subset = multi[multi['n_volunteers'] == nv]['rel_std'].dropna()
        if len(subset) >= 5:
            fracs.append((subset < thresh).mean())
        else:
            fracs.append(np.nan)
    axes[2].plot(list(n_vol_range), fracs, 'o-', color=col, label=f'rel_std < {thresh}', linewidth=1.5)
axes[2].set_xlabel('Number of volunteers per group')
axes[2].set_ylabel('Fraction of groups below threshold')
axes[2].set_title('Quality vs coverage tradeoff')
axes[2].legend(fontsize=9)
axes[2].set_ylim(0, 1)

plt.suptitle('Within-group spread analysis (primary accepted, no bias correction)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_group_stability.png'), dpi=150)
plt.show()
print("Saved: plot_group_stability.png")

# CHANGE 3: Per-group outlier detection

df_accepted_work = df_raw[df_raw['volunteer_id'].isin(accepted_ids)].copy()
df_accepted_work = df_accepted_work.groupby(['volunteer_id', 'day_id', 'group_id'])['spot_count'].mean().reset_index()

df_with_stats = df_accepted_work.merge(
    gc_none[['day_id', 'group_id', 'mean_count', 'std_count', 'n_volunteers']],
    on=['day_id', 'group_id']
)
df_with_stats = df_with_stats[df_with_stats['n_volunteers'] >= 3].copy()
df_with_stats['deviation'] = df_with_stats['spot_count'] - df_with_stats['mean_count']
df_with_stats['z_score'] = df_with_stats['deviation'] / df_with_stats['std_count'].replace(0, np.nan)

for z_thresh in [2.0, 2.5, 3.0]:
    n_outliers = (df_with_stats['z_score'].abs() > z_thresh).sum()
    total = df_with_stats['z_score'].notna().sum()
    print(f"Z > {z_thresh}: {n_outliers} outlier classifications out of {total} ({100*n_outliers/total:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

z_clean = df_with_stats['z_score'].dropna()
axes[0].hist(z_clean.clip(-5, 5), bins=100, color='steelblue', alpha=0.8, edgecolor='none', density=True)
x = np.linspace(-5, 5, 200)
from scipy.stats import norm
axes[0].plot(x, norm.pdf(x), 'r-', linewidth=2, label='Standard normal')
axes[0].axvline(2, color='orange', linestyle='--', label='z = ±2')
axes[0].axvline(-2, color='orange', linestyle='--')
axes[0].axvline(3, color='red', linestyle='--', label='z = ±3')
axes[0].axvline(-3, color='red', linestyle='--')
axes[0].set_xlabel('Z-score (deviation from group mean / group std)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of individual count deviations\nCompared to Gaussian expectation')
axes[0].legend()

sample = df_with_stats.sample(min(20000, len(df_with_stats)), random_state=42)
outlier_mask = sample['z_score'].abs() > 2
axes[1].scatter(sample.loc[~outlier_mask, 'mean_count'], sample.loc[~outlier_mask, 'z_score'],
                alpha=0.05, s=3, color='steelblue', label='Normal')
axes[1].scatter(sample.loc[outlier_mask, 'mean_count'], sample.loc[outlier_mask, 'z_score'],
                alpha=0.3, s=8, color='tomato', label='Outlier (|z|>2)')
axes[1].axhline(2, color='orange', linestyle='--', linewidth=1)
axes[1].axhline(-2, color='orange', linestyle='--', linewidth=1)
axes[1].set_xlabel('Group mean count')
axes[1].set_ylabel('Z-score')
axes[1].set_title('Where do outliers occur?')
axes[1].set_xlim(0, 80)
axes[1].set_ylim(-8, 8)
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plot_outlier_detection.png'), dpi=150)
plt.show()
print("Saved: plot_outlier_detection.png")
